## SWE vs the World: Cross-Country Summary

Where is the SWE premium largest? PPP-adjusted comparison of SWE's pay advantage
over each peer role across USA, India, and China.

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import plotly.express as px

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}
country_colors = {'USA': '#2171b5', 'India': '#31a354', 'China': '#e6550d'}

usa = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
india = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_india_data.csv')
china = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_china_data.csv')
df = pd.concat([usa, india, china], ignore_index=True)
mid = df[df['career_stage'] == 'mid'].copy()
mid['role_label'] = mid['role'].map(ROLE_LABELS)

In [ ]:
swe_ppp = mid[mid['role'] == 'software_engineer'][['country', 'median_salary_ppp_usd']].rename(
    columns={'median_salary_ppp_usd': 'swe_ppp'}
)
peers = mid[mid['role'] != 'software_engineer'].merge(swe_ppp, on='country')
peers['swe_multiple_ppp'] = (peers['swe_ppp'] / peers['median_salary_ppp_usd']).round(2)
peers['role_label'] = peers['role'].map(ROLE_LABELS)

usa_order = (
    peers[peers['country'] == 'USA']
    .sort_values('swe_multiple_ppp', ascending=False)['role_label'].tolist()
)

fig = px.bar(
    peers, x='role_label', y='swe_multiple_ppp', color='country',
    barmode='group',
    category_orders={'role_label': usa_order, 'country': ['USA', 'India', 'China']},
    color_discrete_map=country_colors,
    title='SWE Pay Multiple vs Peer Roles by Country (PPP-Adjusted, Mid-Career, 2023)<br>'
          '<sup>Higher = SWE earns proportionally more than this role in that country</sup>',
    labels={'swe_multiple_ppp': 'SWE / Role PPP Salary Multiple', 'role_label': 'Role', 'country': 'Country'},
)
fig.add_hline(y=1.0, line_dash='dot', line_color='red', annotation_text='Equal pay')
fig.update_layout(xaxis_tickangle=-35)
fig.show()

In [ ]:
heat = peers.pivot_table(index='role_label', columns='country', values='swe_multiple_ppp')

fig2 = px.imshow(
    heat, text_auto=True, aspect='auto',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=1.0,
    title='SWE Salary Multiple by Role and Country (PPP-Adjusted, Mid-Career)',
    labels={'color': 'SWE / Role multiple'},
)
fig2.show()

In [ ]:
farm_ppp = mid[mid['role'] == 'farm_worker'][['country', 'median_salary_ppp_usd']].rename(
    columns={'median_salary_ppp_usd': 'farm_ppp'}
)
swe_vs_farm = swe_ppp.merge(farm_ppp, on='country')
swe_vs_farm['multiple'] = (swe_vs_farm['swe_ppp'] / swe_vs_farm['farm_ppp']).round(1)

fig3 = px.bar(
    swe_vs_farm, x='country', y='multiple', color='country',
    color_discrete_map=country_colors,
    title='SWE Mid-Career Salary vs Farm Worker — PPP Multiple (2023)<br>'
          '<sup>A software engineer earns X times what a farm worker earns</sup>',
    labels={'multiple': 'SWE / Farm Worker PPP multiple', 'country': 'Country'},
    text='multiple',
)
fig3.update_traces(texttemplate='%{text}x', textposition='outside')
fig3.show()